In [1]:
import sys, os
import importlib
from importlib import reload
# importlib.import_module(module_name)
from collections import namedtuple
import copy

import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import scipy
import tqdm
import joblib


In [20]:
data_dir_tom="/home/tomren/geant_projects/musim_test/"
data_dir_miriam="/project/rrg-mdiamond/data/MATHUSLA/simulation_v2/"
data_dir_heather="/project/def-hrussell/MATHUSLA/simulation/"
data_dir = data_dir_miriam

# Load data

In [21]:
efficiency=0.95
data_dir = data_dir_miriam
filenames = {\
    "bkg p": f"{data_dir_miriam}/cosmic/backup_cosmic_p/skim//rrq_bkg_eff{efficiency:.2f}.joblib",\
    "bkg n": f"{data_dir_miriam}/cosmic/backup_cosmic_n_partial/skim//rrq_bkg_eff{efficiency:.2f}.joblib",\
    "bkg mu_c": f"{data_dir}/cosmic/cosmic_mu/merged/rrq_bkg_eff{efficiency:.2f}.joblib",
    "bkg n+p": f"{data_dir}/cosmic/cosmic_np/merged/rrq_bkg_eff{efficiency:.2f}.joblib",
}

exposure = {\
    "bkg p":1/32,
    "bkg n":1/27,
    "bkg mu":1,
    "bkg v":20,
    "sig 15":1,
    "sig 25":1,
    "sig 35":1,
    "sig 45":1,
    "sig 55":1,
    "bkg v_l":3000*2,
    "bkg mu_c":1/29.5, # 61.8 days
    "bkg n+p":1/27.3, # 66.67 days
} # Fraction of 5 years

In [25]:
## Load raw data
raw = {item: joblib.load(filenames[item]) for item in filenames}

In [44]:
key0 = "bkg p"
merged = copy.copy(raw[key0])
seed = 0
rng = np.random.default_rng(seed)

In [45]:
merged.keys()

dict_keys(['Run_number', 'Evt_number', 'ROOT_entry', 'gen_p3', 'gen_xyzt', 'gen_pdgID', 'event_ntracks', 'event_nhits', 'event_nvertices', 'event_ntrack_reconstructable', 'event_ntrack_reconstructable_primary', 'event_vntrk_max', 'event_track_nhits', 'event_track_nhits_upward', 'event_ndigi_veto', 'event_ndigi_active', 'vertex_topfrac', 'vertex_xyzt', 'vertex_ntracks', 'vertex_ndigi', 'vertex_chi2', 'vertex_prob', 'vertex_residual', 'vertex_error', 'vertex_residual_longitrans', 'vertex_ntracklet_0', 'vertex_ntracklet_2', 'vertex_ntracklet_3+', 'vertex_ndownward_track', 'event_ndownward_track', 'vertex_ndigi_veto_before_limited', 'vertex_ndigi_active_before_limited', 'vertex_slowest_track', 'vertex_ndigi_veto_after', 'vertex_ndigi_veto_after_comp', 'vertex_ndigi_active_after', 'vertex_ndigi_active_after_comp', 'vertex_open_angle', 'vertex_cms_angle_h', 'vertex_cms_angle_v', 'vertex_cms_angle_h_mean', 'vertex_cms_angle_v_mean', 'vertex_cms_angle_h_span', 'vertex_cms_angle_v_span', 'verte

In [46]:
for name in raw:
    if name == key0:
        continue

    data_raw = raw[name]
    counts = len(data_raw["Evt_number"])
    counts_selected = int(exposure[key0]/exposure[name] * counts)
    inds_selected = rng.choice(np.arange(counts),counts_selected , replace=False)
    
    for key in data_raw:
        try:
            merged[key] = np.concatenate((merged[key], data_raw[key][inds_selected]))
        except:
            print(key)

event_vntrk_max
vertex_comp_metric_dt
vertex_comp_metric_speed
event_vntrk_max
vertex_track_dist
vertex_comp_metric_dt
vertex_comp_metric_speed
vertex_track_to_veto_dist
event_vntrk_max
vertex_track_dist
vertex_comp_metric_dt
vertex_comp_metric_speed
vertex_track_to_veto_dist


In [47]:
len(merged["Evt_number"])


860795

In [48]:
joblib.dump(merged, f"{data_dir}/cosmic/merged_rrq_bkg_eff{efficiency:.2f}.joblib")

['/project/rrg-mdiamond/data/MATHUSLA/simulation_v2//cosmic/merged_rrq_bkg_eff0.95.joblib']